# NB01 · Semana 2 — Recorrido del stack de datos

**Ingeniería de Datos — LCD · UCA Rosario · 2026**  
Versión: **para el alumno (a completar)**

## Objetivo

Conocer los cinco componentes del stack y verificar que responden. Todavía no se programa el pipeline: se trata de entender **qué hace cada pieza** y cómo se corresponde con el ciclo de vida de la Ingeniería de Datos.

| Componente | Etapa | En Docker local | En el servidor de la cátedra |
|---|---|---|---|
| MinIO | Data Lake (S3) | http://localhost:9001 | acceso por notebook (bucket del grupo) |
| DuckDB | Transformación | en el notebook | en el notebook |
| Airflow | Orquestación | http://localhost:8080 | https://airflow.nexoacademia.org |
| PostgreSQL | Serving (gold) | localhost:5432 | interno (por notebook) |
| Metabase | BI | http://localhost:3000 | https://metabase.nexoacademia.org |
| Grafana | Observabilidad | (no está en el mini-stack local) | https://grafana.nexoacademia.org |

La celda de configuración de abajo detecta sola en cuál de los dos entornos estás.

In [ ]:
# === Configuración del entorno — NO tocar ===
# El MISMO notebook corre en el Docker local y en el servidor de la cátedra.
# entorno.py lee todo de variables de entorno, con valores por defecto locales.
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import (BUCKET, BRONZE, SILVER, STORAGE_OPTIONS,
                     PG, pg_url, SEED, DBT_DIR, DAGS_DIR, info)
info()

In [ ]:
STORAGE_OPTIONS

## 1. Verificar el lake (MinIO)

Conectamos al object storage por el protocolo S3 y listamos los buckets a los que tenemos acceso. En local existe el bucket `lake`; en el servidor, el bucket de tu grupo.

In [ ]:
import boto3
# TODO(alumno): crear el cliente S3 usando los valores de STORAGE_OPTIONS
# (AWS_ENDPOINT_URL, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY) y listar los buckets.
s3 = boto3.client('s3', endpoint_url=STORAGE_OPTIONS["AWS_ENDPOINT_URL"], aws_access_key_id=STORAGE_OPTIONS["AWS_ACCESS_KEY_ID"], aws_secret_access_key=STORAGE_OPTIONS["AWS_SECRET_ACCESS_KEY"])


In [ ]:
s3

## 2. Verificar DuckDB

DuckDB es un motor analítico *in-process*: no hay servidor, corre dentro de este notebook. Lo usaremos para transformar datos del lake.

In [ ]:
import duckdb
print('DuckDB', duckdb.__version__)
print(duckdb.sql('SELECT 42 AS respuesta, current_date AS hoy').df())

## 3. Verificar el warehouse de serving (PostgreSQL)

Es la base donde luego publicaremos la capa **gold** para que Metabase la consuma. La URL de conexión la arma `pg_url()` según el entorno.

In [ ]:
from sqlalchemy import create_engine, text
# Crear conexión a PostgreSQL usando la URL definida en entorno.py
engine = create_engine(pg_url())
# Abrir la conexión y consultar la versión de PostgreSQL
with engine.connect() as con:
    version = con.execute(text("SELECT version()")).scalar()
    print(version)


## Cierre

Cada pieza del stack cumple una etapa del ciclo de vida. En las próximas clases construiremos el pipeline que las recorre: extracción → lake → transformación → serving → BI.

> **Nota:** este notebook asume el stack levantado. En local: `make up`. En el servidor ya está corriendo.